# Trabajo de Aprendizaje Profundo - Reconocimiento de Tumores

Realizado por Teresa Valero Díaz y Julia Ballati Martínez

## 1. Carga de los datos

In [12]:
import zipfile
import os

# Usamos una ruta relativa (sin la barra inicial /)
local_zip = './content/bbdd_tumors.zip' 
random_state=42 # para la reproducibilidad
if os.path.exists(local_zip):
    with zipfile.ZipFile(local_zip, 'r') as zip_ref:
        zip_ref.extractall('./content')
        zip_ref.close()
        base_dir = './content/Data'
else:
    print(f"Error: No existe la ruta {os.path.abspath(local_zip)}")

In [13]:
# Estas son los nombres de las carpetas que hay en el zip, y se corresponden con las 4 clases que hay
clases = ['pituitary_tumor', 'meningioma_tumor', 'glioma_tumor', 'normal']

In [14]:
import splitfolders
from pathlib import Path
input = Path(base_dir) 
output = Path(".") / "data"
train_size = 0.6   
val_size = 0.2 
test_size = 0.2 
if not os.path.exists(output):
    splitfolders.ratio(input, output=output, ratio=(train_size, val_size,test_size), seed=random_state)
else:
    print("Directory already exists.")

Copying files: 3096 files [00:01, 2314.97 files/s]


## 2. Análisis del tamaño de las imágenes

El tamaño de todas las imágenes debería ser de 256 x 256, lo cual comprobaremos a continuación

In [15]:
from PIL import Image
import pandas as pd

train_path = "./data/train"
original_sizes = []

for root, dirs, files in os.walk(train_path):
    for file in files:
        if file.endswith(('.png', '.jpg', '.jpeg')):
            with Image.open(os.path.join(root, file)) as img:
                original_sizes.append(img.size) # size es (ancho, alto)

df_sizes = pd.DataFrame(original_sizes, columns=['Ancho', 'Alto'])
print("Tamaños de las imágenes:")
print(df_sizes.value_counts())

Tamaños de las imágenes:
Ancho  Alto
256    256     1855
Name: count, dtype: int64


In [16]:
from tensorflow import keras
batch_size = 32  
image_size = (256,256)
directory = Path(".") / "data"/ "train"  # Directorio donde se ubican los datos
# Genera los datasets de train, validación y test a partir de las imágenes de los directorios
ds_train = keras.utils.image_dataset_from_directory(directory, batch_size = batch_size, image_size = image_size, seed = random_state)
directory = Path(".") / "data" / "val"
ds_validation = keras.utils.image_dataset_from_directory(directory, batch_size = batch_size, image_size = image_size, seed = random_state)
directory = Path(".") / "data" / "test"
ds_test = keras.utils.image_dataset_from_directory(directory, batch_size = batch_size, image_size = image_size, shuffle = False)

Found 1855 files belonging to 4 classes.
Found 617 files belonging to 4 classes.
Found 624 files belonging to 4 classes.
